# Run C sanity check: generator-last-p reference vs Taillard C++

This notebook checks whether our Run C reference builder is aligned with Prof. Taillard's `clustering_sphere.cpp`.

It focuses only on **instance_id = 1** instances.

It does three things:

1. Loads selected `cluster_tai` instances with `instance_id = 1`.
2. Computes the absolute Run C metric in Python using the **last `p` points as reference centers**.
3. Runs Taillard's C++ program on the same instances/options and compares the printed C++ `Reference` value to the Python value.

Run C metric:

\[
F(C) = \sum_j r_j^d,\quad r_j = \max_{x_i \in S_j} \lVert x_i - c_j Vert_2
\]

For the generator reference, `C` is the set of the **last `p` points** of the instance file.

Important distinction:

- The **reference sanity check** compares the same solution in Python and C++: `centers = X[-p:]`.
- The C++ option outputs are extra baseline results. Their ratios are comparable only because they use the same reference denominator.


In [ ]:
# ============================================================
# CONTROL PANEL
# ============================================================

from pathlib import Path

# Main Drive folder
TM_DIR = Path("/content/drive/MyDrive/TM")

# Required files
CLUSTER_ZIP_PATH = TM_DIR / "cluster_tai.zip"
CPP_PATH = TM_DIR / "clustering_sphere.cpp"

# Working folder inside Colab runtime
WORK_DIR = Path("/content/runC_taillard_reference_sanity")
EXTRACT_DIR = WORK_DIR / "instances"
CPP_BIN = WORK_DIR / "clustering_sphere"

# This notebook intentionally sticks to instance_id = 1.
INSTANCE_ID = 1

# Multiple sizes/dimensions, all with instance_id = 1.
# n is p^2 in these Taillard synthetic filenames.
# Keep this list small at first; option 0/1 can be slow in high dimension.
TEST_SPECS = [
    {"d": 2, "p": 15},
    {"d": 2, "p": 20},
    {"d": 2, "p": 70},
    {"d": 3, "p": 40},
    {"d": 4, "p": 20},
    {"d": 4, "p": 70},
]


def cluster_tai_name_from_spec(d: int, p: int, instance_id: int = INSTANCE_ID) -> str:
    n = int(p) * int(p)
    return f"cluster_tai{n:05d}_{int(p):03d}_{int(d)}_{int(instance_id)}.csv"


INSTANCE_NAMES = [
    cluster_tai_name_from_spec(spec["d"], spec["p"], INSTANCE_ID)
    for spec in TEST_SPECS
]

# Test all 3 options when feasible:
# 0 = kmedian-like refinement
# 1 = PAM-style method, expensive
# 2 = hybrid PAM-on-sample + refinement
OPTIONS_TO_TEST = [0, 1, 2]

# To avoid accidentally running full PAM on large instances.
RUN_OPTION_1_PAM_ONLY_IF_N_LEQ = 400

# Number of C++ runs per method call.
# Use 1 for deterministic/reference sanity; increase later for baseline statistics.
CPP_RUNS = 1

# Timeout per C++ call.
# Even if a method times out, the C++ program often prints the Reference first;
# this notebook still parses that partial output.
CPP_TIMEOUT_S = 300

# Distance batch size for Python metric computation.
DISTANCE_BATCH_SIZE = 2048

print("CLUSTER_ZIP_PATH:", CLUSTER_ZIP_PATH)
print("CPP_PATH:", CPP_PATH)
print("WORK_DIR:", WORK_DIR)
print("INSTANCE_ID:", INSTANCE_ID)
print("INSTANCE_NAMES:")
for name in INSTANCE_NAMES:
    print(" -", name)


In [ ]:
# ============================================================
# Mount Drive, check files
# ============================================================

import os
import subprocess
import sys
import zipfile
import re
import math
import time
import shutil
import json
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("[info] Drive mount skipped or unavailable:", repr(e))

missing = []
for p in [CLUSTER_ZIP_PATH, CPP_PATH]:
    ok = Path(p).exists()
    print(("OK      " if ok else "MISSING "), p)
    if not ok:
        missing.append(str(p))

if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

WORK_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print("Ready.")


In [ ]:
# ============================================================
# Extract selected instances from cluster_tai.zip
# ============================================================

with zipfile.ZipFile(CLUSTER_ZIP_PATH, "r") as z:
    available = set(z.namelist())

    for name in INSTANCE_NAMES:
        if name not in available:
            # Allow path-inside-zip fallback
            matches = [x for x in available if x.endswith("/" + name) or x.endswith(name)]
            if not matches:
                raise FileNotFoundError(f"{name} not found in {CLUSTER_ZIP_PATH}")
            zip_member = matches[0]
        else:
            zip_member = name

        out_path = EXTRACT_DIR / Path(name).name
        out_path.write_bytes(z.read(zip_member))
        print("Extracted:", zip_member, "->", out_path)

print("Extracted files:", len(list(EXTRACT_DIR.glob("*.csv"))))


In [ ]:
# ============================================================
# Compile Taillard C++ code
# ============================================================

cpp_local = WORK_DIR / "clustering_sphere.cpp"
shutil.copy2(CPP_PATH, cpp_local)

compile_cmds = [
    ["g++", "-O3", "-std=c++17", str(cpp_local), "-o", str(CPP_BIN)],
    ["g++", "-O3", "-std=c++11", str(cpp_local), "-o", str(CPP_BIN)],
]

last_err = None
for cmd in compile_cmds:
    print("Compiling:", " ".join(cmd))
    res = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if res.returncode == 0:
        print("Compiled OK:", CPP_BIN)
        break
    last_err = res.stderr
    print("Compile failed with this command.")
else:
    raise RuntimeError("Could not compile C++ file.\nLast stderr:\n" + str(last_err))

print("Binary exists:", CPP_BIN.exists())


In [ ]:
# ============================================================
# Python loader + Run C metric
# ============================================================

def load_cluster_tai_instance(path: Path):
    """
    File format:
      first line: n p d
      next n lines: coordinates
    In the generator-reference interpretation, the last p points are the generator centers.
    """
    path = Path(path)
    with open(path, "r", encoding="utf-8") as f:
        first = f.readline().strip().split()
        if len(first) < 3:
            raise ValueError(f"Bad header in {path}: {first}")
        n, p, d = map(int, first[:3])

    X = np.loadtxt(path, skiprows=1, dtype=float)
    if X.ndim == 1:
        X = X.reshape(1, -1)

    if X.shape != (n, d):
        raise ValueError(f"Expected X shape {(n, d)}, got {X.shape} for {path}")

    generator_centers = X[-p:].copy()
    return {
        "name": path.name,
        "n": n,
        "p": p,
        "d": d,
        "X": X,
        "generator_centers": generator_centers,
    }


def radius_volume_cost(X, centers, d, batch_size=2048):
    """
    Computes sum_j max_{assigned i to j} ||x_i - c_j||^d.

    It uses squared distances internally:
      ||x-c||^d = (||x-c||^2)^(d/2)
    """
    X = np.asarray(X, dtype=float)
    centers = np.asarray(centers, dtype=float)

    if centers.ndim != 2:
        raise ValueError(f"centers must be 2D, got {centers.shape}")

    n = X.shape[0]
    k = centers.shape[0]

    max_sq_by_center = np.zeros(k, dtype=float)

    for start in range(0, n, batch_size):
        xb = X[start:start + batch_size]
        sq = ((xb[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
        labels = np.argmin(sq, axis=1)
        min_sq = sq[np.arange(len(xb)), labels]

        for j in np.unique(labels):
            local_max = float(min_sq[labels == j].max())
            if local_max > max_sq_by_center[j]:
                max_sq_by_center[j] = local_max

    return float(np.sum(max_sq_by_center ** (float(d) / 2.0)))


def generator_reference_cost(path: Path):
    inst = load_cluster_tai_instance(path)
    X = inst["X"]
    C = inst["generator_centers"]
    d = inst["d"]

    # Main interpretation: all n rows are points to cover, and the last p rows are also reference centers.
    cost_include_last_p = radius_volume_cost(X, C, d, batch_size=DISTANCE_BATCH_SIZE)

    # Alternative sanity check: cover only the first n-p generated elements.
    # This is NOT expected to match Taillard C++ if his code assigns all n rows.
    cost_exclude_last_p = radius_volume_cost(X[:-inst["p"]], C, d, batch_size=DISTANCE_BATCH_SIZE)

    return {
        "instance": inst["name"],
        "n": inst["n"],
        "p": inst["p"],
        "d": inst["d"],
        "python_ref_include_last_p": cost_include_last_p,
        "python_ref_exclude_last_p": cost_exclude_last_p,
    }


py_ref_rows = []
for name in INSTANCE_NAMES:
    path = EXTRACT_DIR / Path(name).name
    row = generator_reference_cost(path)
    py_ref_rows.append(row)

py_ref_df = pd.DataFrame(py_ref_rows)
display(py_ref_df)


In [ ]:
# ============================================================
# Run Taillard C++ and parse/reference-check output
# ============================================================

FLOAT_RE = re.compile(r"[-+]?(?:\d+\.\d*|\.\d+|\d+)(?:[eE][-+]?\d+)?")


def ensure_text(x):
    """subprocess.TimeoutExpired can carry bytes even when text=True; normalize safely."""
    if x is None:
        return ""
    if isinstance(x, bytes):
        return x.decode("utf-8", errors="replace")
    return str(x)


def parse_all_floats(text: str):
    text = ensure_text(text)
    vals = []
    for m in FLOAT_RE.finditer(text):
        try:
            vals.append(float(m.group(0)))
        except Exception:
            pass
    return vals


def parse_labelled_reference(text: str):
    """
    Parse Taillard's line:
      Reference (value, time[s]): <value> <time>
    """
    text = ensure_text(text)
    patterns = [
        r"(?i)Reference\s*\(value,\s*time\[s\]\)\s*:\s*(" + FLOAT_RE.pattern + r")",
        r"(?i)reference[^0-9eE+\-]*(" + FLOAT_RE.pattern + r")",
        r"(?i)ref(?:erence)?[_ ]?cost[^0-9eE+\-]*(" + FLOAT_RE.pattern + r")",
    ]
    for pat in patterns:
        m = re.search(pat, text)
        if m:
            return float(m.group(1))
    return None


def parse_reference_improved_ratio(text: str):
    """
    Parse line:
      Reference improved with PAM (value/reference, time[s]): <ratio> <time>
    This is not the selected option result; it is an extra reference-improvement line.
    """
    text = ensure_text(text)
    pat = r"(?i)Reference improved with PAM.*?:\s*(" + FLOAT_RE.pattern + r")\s+(" + FLOAT_RE.pattern + r")"
    m = re.search(pat, text)
    if not m:
        return None, None
    return float(m.group(1)), float(m.group(2))


def parse_final_method_ratio(text: str):
    """
    Taillard's program prints the selected option result as a final line like:
      <value/reference> <time>
    We parse the last non-empty line containing exactly two floats and not starting with Reference.
    """
    text = ensure_text(text)
    candidates = []
    for line in text.splitlines():
        stripped = line.strip()
        if not stripped:
            continue
        if stripped.lower().startswith("reference"):
            continue
        vals = parse_all_floats(stripped)
        if len(vals) == 2:
            candidates.append((vals[0], vals[1], stripped))
    if not candidates:
        return None, None, None
    ratio, runtime_s, raw_line = candidates[-1]
    return float(ratio), float(runtime_s), raw_line


def closest_printed_number_to_reference(text: str, py_ref: float):
    nums = parse_all_floats(text)
    if not nums:
        return None, None
    closest = min(nums, key=lambda v: abs(v - py_ref) / max(1.0, abs(py_ref)))
    rel = abs(closest - py_ref) / max(1.0, abs(py_ref))
    return closest, rel


def run_cpp(instance_path: Path, option: int, runs: int, timeout_s: int):
    cmd = [str(CPP_BIN), str(instance_path), str(option), str(runs)]
    t0 = time.perf_counter()
    try:
        res = subprocess.run(
            cmd,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=timeout_s,
        )
        elapsed = time.perf_counter() - t0
        return {
            "returncode": res.returncode,
            "elapsed_s": elapsed,
            "stdout": ensure_text(res.stdout),
            "stderr": ensure_text(res.stderr),
            "timeout": False,
            "cmd": " ".join(cmd),
        }
    except subprocess.TimeoutExpired as e:
        elapsed = time.perf_counter() - t0
        return {
            "returncode": None,
            "elapsed_s": elapsed,
            "stdout": ensure_text(e.stdout),
            "stderr": ensure_text(e.stderr),
            "timeout": True,
            "cmd": " ".join(cmd),
        }


cpp_rows = []

for _, ref_row in py_ref_df.iterrows():
    inst_name = ref_row["instance"]
    n = int(ref_row["n"])
    instance_path = EXTRACT_DIR / inst_name
    py_ref = float(ref_row["python_ref_include_last_p"])

    for option in OPTIONS_TO_TEST:
        if option == 1 and n > RUN_OPTION_1_PAM_ONLY_IF_N_LEQ:
            print(f"Skipping option 1/PAM for {inst_name} because n={n} > {RUN_OPTION_1_PAM_ONLY_IF_N_LEQ}")
            continue

        print("
" + "=" * 100)
        print(f"Running C++: instance={inst_name}, option={option}, runs={CPP_RUNS}")
        result = run_cpp(instance_path, option=option, runs=CPP_RUNS, timeout_s=CPP_TIMEOUT_S)

        combined_output = ensure_text(result["stdout"]) + "
" + ensure_text(result["stderr"])

        print("Command:", result["cmd"])
        print("Timeout:", result["timeout"], "returncode:", result["returncode"], "elapsed_s:", round(result["elapsed_s"], 3))
        if result["stdout"].strip():
            print("--- stdout ---")
            print(result["stdout"][:4000])
        if result["stderr"].strip():
            print("--- stderr ---")
            print(result["stderr"][:2000])

        labelled_ref = parse_labelled_reference(combined_output)
        pam_ref_ratio, pam_ref_runtime = parse_reference_improved_ratio(combined_output)
        method_ratio, method_runtime, method_raw_line = parse_final_method_ratio(combined_output)
        closest_num, closest_rel = closest_printed_number_to_reference(combined_output, py_ref)

        cpp_ref_rel_diff = np.nan
        if labelled_ref is not None:
            cpp_ref_rel_diff = abs(labelled_ref - py_ref) / max(1.0, abs(py_ref))

        # If the C++ printed ratio uses its own reference denominator, convert it to an absolute value.
        # If cpp reference == python reference, then method_gap_vs_python_ref_pct == 100*(method_ratio-1).
        method_abs_from_cpp_ratio = np.nan
        method_gap_vs_python_ref_pct = np.nan
        if labelled_ref is not None and method_ratio is not None:
            method_abs_from_cpp_ratio = method_ratio * labelled_ref
            method_gap_vs_python_ref_pct = 100.0 * (method_abs_from_cpp_ratio / py_ref - 1.0)

        cpp_rows.append({
            "instance": inst_name,
            "n": n,
            "p": int(ref_row["p"]),
            "d": int(ref_row["d"]),
            "option": option,
            "runs": CPP_RUNS,
            "timeout": result["timeout"],
            "returncode": result["returncode"],
            "elapsed_s": result["elapsed_s"],
            "python_ref_include_last_p": py_ref,
            "python_ref_exclude_last_p": float(ref_row["python_ref_exclude_last_p"]),
            "cpp_labelled_reference": labelled_ref,
            "cpp_ref_rel_diff_vs_python": cpp_ref_rel_diff,
            "cpp_reference_matches_python": bool(cpp_ref_rel_diff <= 1e-9) if not pd.isna(cpp_ref_rel_diff) else False,
            "cpp_pam_improved_reference_ratio": pam_ref_ratio,
            "cpp_pam_improved_reference_runtime_s": pam_ref_runtime,
            "cpp_method_ratio": method_ratio,
            "cpp_method_runtime_s": method_runtime,
            "cpp_method_raw_line": method_raw_line,
            "cpp_method_abs_cost_from_ratio": method_abs_from_cpp_ratio,
            "cpp_method_gap_vs_python_ref_pct": method_gap_vs_python_ref_pct,
            "closest_printed_number_to_python_ref": closest_num,
            "closest_printed_number_rel_diff": closest_rel,
            "stdout_first_1000": result["stdout"][:1000],
            "stderr_first_1000": result["stderr"][:1000],
        })

cpp_df = pd.DataFrame(cpp_rows)
display(cpp_df)


In [ ]:
# ============================================================
# Interpret comparison
# ============================================================

summary = cpp_df.copy()

cols = [
    "instance", "n", "p", "d", "option", "timeout", "returncode", "elapsed_s",
    "python_ref_include_last_p",
    "python_ref_exclude_last_p",
    "cpp_labelled_reference",
    "cpp_ref_rel_diff_vs_python",
    "cpp_reference_matches_python",
    "cpp_pam_improved_reference_ratio",
    "cpp_method_ratio",
    "cpp_method_abs_cost_from_ratio",
    "cpp_method_gap_vs_python_ref_pct",
    "cpp_method_runtime_s",
]

existing_cols = [c for c in cols if c in summary.columns]
display(summary[existing_cols])

print("
Interpretation:")
print("- Main sanity check: cpp_labelled_reference must match python_ref_include_last_p.")
print("- If cpp_reference_matches_python is True, our generator-last-p Run C reference matches Taillard's Reference.")
print("- cpp_method_ratio is the C++ option result: method_cost / C++ Reference.")
print("- cpp_method_gap_vs_python_ref_pct converts that same option result to our gap convention using Python's reference.")
print("- Different C++ options are not used to validate the reference; they are baseline methods once the reference denominator is confirmed.")

bad = summary[
    summary["cpp_labelled_reference"].notna()
    & (summary["cpp_ref_rel_diff_vs_python"] > 1e-9)
]
if len(bad) == 0:
    print("
OK: all parsed C++ Reference values match the Python last-p reference within tolerance.")
else:
    print("
WARNING: some C++ Reference values do not match the Python last-p reference:")
    display(bad[["instance", "option", "python_ref_include_last_p", "cpp_labelled_reference", "cpp_ref_rel_diff_vs_python"]])


In [ ]:
# ============================================================
# Save sanity-check results
# ============================================================

out_dir = TM_DIR / "llm-clustering-runs" / "runC_reference_sanity_checks"
out_dir.mkdir(parents=True, exist_ok=True)

timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
py_ref_path = out_dir / f"python_generator_reference_{timestamp}.csv"
cpp_cmp_path = out_dir / f"taillard_cpp_reference_comparison_{timestamp}.csv"

py_ref_df.to_csv(py_ref_path, index=False)
cpp_df.to_csv(cpp_cmp_path, index=False)

print("Saved:")
print(py_ref_path)
print(cpp_cmp_path)
